# DCFA custom continuous-treatment analysis

This notebook runs one bounded DCFA workflow in your own ephemeral Colab runtime. It accepts exactly one continuous outcome Y, one continuous treatment X, and one scalar instrument Z, with 120–256 rows and no baseline covariates. It is not production causal advice, proof that an instrument is valid, or locked Track T evidence.

You supply your own Google Gemini and Prior Labs accounts. Your question is sent to Google; after a separate confirmation, the selected Y/X/Z rows and prediction grids are sent to Prior Labs. Do not use sensitive, confidential, personally identifiable, or unshareable data. Provider availability, quota, charges, Colab resources, and free usage are not guaranteed. This notebook does not launch Gradio, a tunnel, SSH, or a public application server.

## 1. Install the frozen release
The notebook file is opened from `main`, but the runtime below installs one exact approved DCFA commit and checks its statistical source identity.

In [ ]:
import subprocess
import sys

DCFA_RELEASE_COMMIT = "87b2b750d1c9a83497f5b16a7b0597758214d20a"
DCFA_EXPECTED_SOURCE_SHA256 = "sha256:e1c86e9e7bd470f6f6e5c9d5ef9beaad8161e653355002d4ca47555484879791"
DCFA_PROFILE = "website_demo_gemini_v1 + tabpfn_client_managed_demo_v2"
requirements_url = f"https://raw.githubusercontent.com/GepingChen/DCFA/{DCFA_RELEASE_COMMIT}/requirements-website-demo.lock"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_url], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/GepingChen/DCFA.git@{DCFA_RELEASE_COMMIT}", "--no-deps"], check=True)
from dcfa.provenance import dcfa_source_tree_hash

if dcfa_source_tree_hash() != DCFA_EXPECTED_SOURCE_SHA256:
    raise RuntimeError("Installed DCFA source identity does not match this notebook release.")
print(f"DCFA release {DCFA_RELEASE_COMMIT[:12]} is installed with the approved profile.")

## 2. Read your Colab Secrets
Create private Colab Secrets named `DCFA_GEMINI_API_KEY` and `DCFA_TABPFN_TOKEN`, enable notebook access for each, then run this cell. The values are read only now and are never printed. New Gemini keys should follow Google's current authorization-key guidance.

In [ ]:
from google.colab import userdata

try:
    GEMINI_SECRET = userdata.get("DCFA_GEMINI_API_KEY")
    TABPFN_SECRET = userdata.get("DCFA_TABPFN_TOKEN")
except Exception:
    raise RuntimeError("Both named Colab Secrets must exist and allow notebook access.") from None
if not GEMINI_SECRET or not TABPFN_SECRET:
    raise RuntimeError("Both named Colab Secrets must be nonempty.")
print("Two user-owned provider secrets are ready. Secret values were not displayed.")

## 3. Upload and preflight one CSV
Upload one UTF-8 CSV containing exactly the three selected numeric columns. This cell performs local validation only and makes no provider request.

In [ ]:
from pathlib import Path
from google.colab import files
from dcfa_colab import preflight_colab_csv

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one CSV file.")
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
CSV_PATH = Path("/content") / Path(uploaded_name).name
CSV_PATH.write_bytes(uploaded_bytes)

OUTCOME_COLUMN = "Y"
TREATMENT_COLUMN = "X"
INSTRUMENT_COLUMN = "Z"
preflight = preflight_colab_csv(CSV_PATH, outcome=OUTCOME_COLUMN, treatment=TREATMENT_COLUMN, instrument=INSTRUMENT_COLUMN)
print(preflight)

## 4. Freeze the question and confirm both transfers
Edit the bounded question if needed. Set each confirmation to `True` only after reviewing its separate destination. Exactly one Gemini request is allowed; there is no retry, LLM bypass, or sklearn fallback.

In [ ]:
QUESTION = "How does the median outcome change between a low and a high supported treatment intervention?"
SEED = 20260813

# Google receives only QUESTION, generic Y/X/Z roles, and symbolic low/center/high labels.
consent_google_transfer = False
# Prior Labs receives the selected Y/X/Z rows and bounded prediction grids.
consent_prior_labs_transfer = False

if not consent_google_transfer or not consent_prior_labs_transfer:
    raise RuntimeError("Review both transfer statements and explicitly set both confirmations to True.")

## 5. Run once, verify, and download
A successful call writes a new immutable runtime directory, independently verifies it without refitting, scans it for both secret values, and creates a downloadable ZIP. A support, provider, or verification failure displays no number.

In [ ]:
from pprint import pprint
from dcfa_colab import run_colab_analysis

RESULT = run_colab_analysis(
    csv_file=CSV_PATH,
    outcome=OUTCOME_COLUMN,
    treatment=TREATMENT_COLUMN,
    instrument=INSTRUMENT_COLUMN,
    question=QUESTION,
    seed=SEED,
    gemini_api_key=GEMINI_SECRET,
    tabpfn_token=TABPFN_SECRET,
    consent_google_transfer=consent_google_transfer,
    consent_prior_labs_transfer=consent_prior_labs_transfer,
    output_root="/content/dcfa-runs",
)
pprint(RESULT.visitor_result)
if RESULT.status == "completed" and RESULT.archive_path is not None:
    files.download(str(RESULT.archive_path))
else:
    print("The run stopped safely. No numerical artifact or download was produced.")

## 6. Clean up the ephemeral runtime
After the download completes, delete the in-memory secret variables and uploaded file below. Then choose **Runtime → Disconnect and delete runtime**. Colab virtual machines are ephemeral, but runtime availability and deletion timing remain controlled by Google.

In [ ]:
del GEMINI_SECRET, TABPFN_SECRET, uploaded_bytes, uploaded
if CSV_PATH.exists():
    CSV_PATH.unlink()
print("Local secret variables and the uploaded CSV were removed. Now use Runtime > Disconnect and delete runtime.")